# Los móviles de ahora no aguantan nada: preparo los datos

**Categoría: EDA (análisis exploratorio de datos).** No hago predicciones.

Quiero comprobar qué parte de esa frase puedo contrastar desde los Nokia de los 2000. Separo autonomía de una carga, desgaste con los ciclos, sustitución de batería y resistencia. No fabrico una nota general de «durabilidad».

## Mi ruta de lectura

Primero explico el origen y la selección; después convierto las mediciones en una tabla; por último reviso su calidad. Continúo en `02_eda_moviles.ipynb`.

**Glosario.** Autonomía: tiempo hasta volver a cargar. Ciclo: consumo acumulado equivalente a una carga completa. Espera: teléfono conectado a la red sin conversación continua. Reparabilidad: facilidad de arreglar o sustituir piezas. Especificación: cifra declarada por una empresa, distinta de una medición independiente.

## 1. Qué selecciono y por qué

Analizo un panel intencional de doce modelos: tres familias (iPhone sin Pro/Max, Galaxy S sin Plus/Ultra y Moto G sin Power) en cuatro cortes: 2015, 2018, 2021 y 2025. Empiezo en 2015 porque la prueba wifi v1.3 se introdujo el 5 de marzo de ese año. Termino en 2025 como último año natural completo; no presento el panel como un ranking de los últimos modelos de 2026. El XR representa la línea general de iPhone en 2018, con una pantalla mayor que la del 6s. El Moto G86 no es un sucesor directo del G30 en precio o prestaciones: comparo ejemplos de una familia, no productos idénticos.

Elijo los modelos por familia y fecha, no por sus horas de batería. Incluyo dos cortes intermedios para evitar contar solo una mejora entre extremos. No es una muestra aleatoria ni representa todas las marcas, las ventas o todos los modelos de estas familias. No puedo convertir tres comparaciones en una media del mercado.

Para los doce modelos conservo la fila original de `WiFi Websurfing` y su pertenencia a `WiFi v1.3`. La prueba documentada navega cada 30 segundos con brillo de unos 150 nits. Comparto la tarea y la referencia de brillo, pero cambian navegadores, sistemas operativos, equipos de red y fechas de ensayo. No es un experimento simultáneo bajo condiciones idénticas. Analizo el conjunto teléfono y software, sin atribuir la mejora a una sola pieza.

No mezclo las horas en espera del Nokia con la navegación. Tampoco junto las dos versiones del test de GSMArena o las notas históricas de iFixit, que han cambiado sus criterios.

In [1]:
# cargo el manifiesto de fuentes para identificar cada modelo y su evidencia
from pathlib import Path
import json
import pandas as pd
from IPython.display import display

raiz = Path.cwd()
fuentes = json.loads((raiz / 'fuentes.json').read_text(encoding='utf-8'))
display(pd.DataFrame([f for f in fuentes if f.get('modelo')])[['modelo','anio','familia','url','archivo']])

,modelo,anio,familia,url,archivo
0,Apple iPhone 6s,2015,iPhone,https://www.notebookcheck.net/Apple-iPhone-6S-...,data/raw/iphone_2015.txt
1,Apple iPhone XR,2018,iPhone,https://www.notebookcheck.net/Apple-iPhone-XR-...,data/raw/iphone_2018.txt
2,Apple iPhone 13,2021,iPhone,https://www.notebookcheck.net/Apple-iPhone-13-...,data/raw/iphone_2021.txt
3,Apple iPhone 17,2025,iPhone,https://www.notebookcheck.net/Apple-iPhone-17-...,data/raw/iphone_2025.txt
4,Samsung Galaxy S6,2015,Galaxy S,https://www.notebookcheck.net/Samsung-Galaxy-S...,data/raw/galaxy_2015.txt
5,Samsung Galaxy S9,2018,Galaxy S,https://www.notebookcheck.net/Samsung-Galaxy-S...,data/raw/galaxy_2018.txt
6,Samsung Galaxy S21,2021,Galaxy S,https://www.notebookcheck.net/Samsung-Galaxy-S...,data/raw/galaxy_2021.txt
7,Samsung Galaxy S25,2025,Galaxy S,https://www.notebookcheck.net/Samsung-Galaxy-S...,data/raw/galaxy_2025.txt
8,Motorola Moto G (2015),2015,Moto G,https://www.notebookcheck.net/Motorola-Moto-G-...,data/raw/moto_2015.txt
9,Motorola Moto G6,2018,Moto G,https://www.notebookcheck.net/Motorola-Moto-G6...,data/raw/moto_2018.txt


## 2. De las filas originales a minutos

Cada fila de la tabla resultante corresponde a un móvil concreto probado por Notebookcheck cuando era nuevo. Conservo modelo, año de lanzamiento, familia, horas originales, minutos, protocolo, URL y fecha de consulta. El año lo identifico en la documentación de cada modelo; no lo deduzco de la fecha de actualización de una reseña.

Extraigo horas y minutos del resultado exacto, en lugar de usar las horas decimales redondeadas de la tabla comparativa. No relleno el vídeo del Galaxy S6: la tabla conservada no contiene ese ensayo. Las declaraciones de Nokia y Apple quedan en otra tabla.

La descarga directa de Notebookcheck devolvió HTTP 403. Por eso he conservado extractos de sus filas numéricas consultadas mediante el buscador web, con su URL y localización de línea. Esos extractos permiten repetir la transformación sin acceso a la web; no son páginas HTML completas. Conservo la procedencia en `fuentes.json`.

In [2]:
# reproduzco la extracción desde las fuentes locales y conservo las dos tablas separadas
from preparar_datos import preparar

datos, declaraciones = preparar(raiz)
display(datos)
display(declaraciones)

,clave,modelo,anio,familia,url,wifi_original,wifi_minutos,video_original,video_minutos,protocolo,consulta,wifi_horas,video_horas
0,galaxy_2015,Samsung Galaxy S6,2015,Galaxy S,https://www.notebookcheck.net/Samsung-Galaxy-S...,7h 55min,475,NaN,NaN,Notebookcheck WiFi v1.3; 150 cd/m²,2026-10-20,7.916667,NaN
1,galaxy_2018,Samsung Galaxy S9,2018,Galaxy S,https://www.notebookcheck.net/Samsung-Galaxy-S...,7h 54min,474,10h 9min,609.0,Notebookcheck WiFi v1.3; 150 cd/m²,2026-10-20,7.900000,10.150000
2,galaxy_2021,Samsung Galaxy S21,2021,Galaxy S,https://www.notebookcheck.net/Samsung-Galaxy-S...,9h 04min,544,15h 41min,941.0,Notebookcheck WiFi v1.3; 150 cd/m²,2026-10-20,9.066667,15.683333
3,galaxy_2025,Samsung Galaxy S25,2025,Galaxy S,https://www.notebookcheck.net/Samsung-Galaxy-S...,18h 02min,1082,26h 30min,1590.0,Notebookcheck WiFi v1.3; 150 cd/m²,2026-10-20,18.033333,26.500000
4,moto_2015,Motorola Moto G (2015),2015,Moto G,https://www.notebookcheck.net/Motorola-Moto-G-...,8h 46min,526,11h 17min,677.0,Notebookcheck WiFi v1.3; 150 cd/m²,2026-10-20,8.766667,11.283333
5,moto_2018,Motorola Moto G6,2018,Moto G,https://www.notebookcheck.net/Motorola-Moto-G6...,9h 13min,553,10h 00min,600.0,Notebookcheck WiFi v1.3; 150 cd/m²,2026-10-20,9.216667,10.000000
6,moto_2021,Motorola Moto G30,2021,Moto G,https://www.notebookcheck.net/Motorola-Moto-G3...,17h 49min,1069,22h 29min,1349.0,Notebookcheck WiFi v1.3; 150 cd/m²,2026-10-20,17.816667,22.483333
7,moto_2025,Motorola Moto G86,2025,Moto G,https://www.notebookcheck.net/The-brightest-st...,17h 18min,1038,20h 08min,1208.0,Notebookcheck WiFi v1.3; 150 cd/m²,2026-10-20,17.300000,20.133333
8,iphone_2015,Apple iPhone 6s,2015,iPhone,https://www.notebookcheck.net/Apple-iPhone-6S-...,8h 12min,492,11h 39min,699.0,Notebookcheck WiFi v1.3; 150 cd/m²,2026-10-20,8.200000,11.650000
9,iphone_2018,Apple iPhone XR,2018,iPhone,https://www.notebookcheck.net/Apple-iPhone-XR-...,15h 10min,910,16h 51min,1011.0,Notebookcheck WiFi v1.3; 150 cd/m²,2026-10-20,15.166667,16.850000


,modelo,magnitud,valor,unidad,tipo,fuente
0,Nokia 3310 (2000),espera,260.0,horas,máximo anunciado,nokia_comunicado.txt
1,Nokia 3310 (2000),conversación,4.5,horas,máximo anunciado,nokia_comunicado.txt
2,iPhone 14 y anteriores,ciclos hasta conservar el 80 %,500.0,ciclos,objetivo de diseño en condiciones ideales,apple_ciclos.txt
3,iPhone 15 y posteriores,ciclos hasta conservar el 80 %,1000.0,ciclos,objetivo de diseño en condiciones ideales,apple_ciclos.txt


## 3. Compruebo la cobertura y los valores ausentes

Compruebo que cada familia tiene cuatro años y cada año tres familias. Busco duplicados y duraciones imposibles. Las huellas SHA-256 permiten comprobar que los extractos de esta copia no han cambiado: no garantizan por sí mismas la veracidad de una fuente externa.

In [3]:
# compruebo la estructura y cuantifico las ausencias sin sustituirlas
display(pd.crosstab(datos['familia'], datos['anio']))
display(datos.isna().sum().to_frame('valores ausentes'))
assert len(datos) == 12
assert datos['wifi_minutos'].notna().all()
assert datos['video_minutos'].isna().sum() == 1
assert datos.duplicated(['familia','anio']).sum() == 0
print('Conservo 12 resultados wifi y 11 resultados de vídeo. No imputo valores.')

anio,2015,2018,2021,2025
familia,,,,
Galaxy S,1,1,1,1
Moto G,1,1,1,1
iPhone,1,1,1,1


,valores ausentes
clave,0
modelo,0
anio,0
familia,0
url,0
wifi_original,0
wifi_minutos,0
video_original,1
video_minutos,1
protocolo,0


Conservo 12 resultados wifi y 11 resultados de vídeo. No imputo valores.


## 4. Qué dejo preparado

He preparado `data/processed/autonomia.csv` y `declaraciones.csv`. En el segundo cuaderno compararé cada familia consigo misma y comprobaré los resultados intermedios. No calcularé años de vida a partir de las horas de navegación, ni transformaré una promesa de ciclos en una experiencia observada.

Conservo las fuentes y las limitaciones completas en el README y en el manifiesto. Para revisar el Nokia original consulto su comunicado de 2000 y las páginas 23-24, 99 y 101 del manual (paginación impresa); su aviso sobre los golpes no demuestra por sí mismo que fuera frágil.